# MuJoCo Basics with MjSpec — Bodies, Joints & Muscles

This tutorial introduces the core building blocks of a MuJoCo simulation by constructing a simple **linked-segment arm** step by step.

We use the Python-native **`MjSpec`** API (available since MuJoCo 3.x) to define models programmatically (i.e., no raw XML file required).  

**What you will learn**
1. How to create a model with `MjSpec` and visualise it interactively with viser
2. What a **body** is and how geoms give it shape
3. How **joints** unlock degrees of freedom (and what happens without them)
4. A first look at **muscles** (tendons + actuators) to drive the model

---

*Tutorial created by: Florian Fischer, University of Cambridge.*

## Setup

In [ ]:
%env MUJOCO_GL=egl

!pip install "mujoco>=3.2" \
    "numpy" \
    "matplotlib"

!command -v ffmpeg >/dev/null || (apt update && apt install -y ffmpeg)
!pip install -q mediapy


In [ ]:
import mujoco
import numpy as np
from pathlib import Path
import logging


# Graphics and plotting.
import mediapy as media
import matplotlib.pyplot as plt
from IPython.display import HTML

print(f"MuJoCo {mujoco.__version__}")

### Visualisation helpers

In [ ]:
def _resolve_fps(
    fps: int | float,
    inputdict: dict[str, str] | None,
    outputdict: dict[str, str] | None,
) -> int | float:
    """Resolve output FPS from explicit arg and skvideo-style dicts."""
    for cfg in (inputdict, outputdict):
        if not cfg:
            continue
        if "-r" in cfg:
            try:
                return float(cfg["-r"])
            except (TypeError, ValueError):
                logging.warning(
                    "Ignoring invalid -r value in video options: %r", cfg["-r"]
                )
    return fps


def write_video(
    video_path: str | Path,
    video_frames: list[np.ndarray] | np.ndarray,
    fps: int | float = 25,
    *,
    inputdict: dict[str, str] | None = None,
    outputdict: dict[str, str] | None = None,
) -> None:
    """Write MP4 frames to disk without scikit-video.

    Args:
        video_path: Destination file path.
        video_frames: Frame sequence with shape ``(T, H, W, 3)`` or list of frames.
        fps: Output frame rate unless overridden by ``inputdict``/``outputdict``.
            Defaults to 25 to match historical ``skvideo.io.vwrite`` behavior.
        inputdict: Backward-compatible skvideo-style ffmpeg options.
        outputdict: Backward-compatible skvideo-style ffmpeg options.
    """
    import imageio

    path = Path(video_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    resolved_fps = _resolve_fps(fps=fps, inputdict=inputdict, outputdict=outputdict)
    codec = (outputdict or {}).get("-vcodec", "libx264")
    pix_fmt = (outputdict or {}).get("-pix_fmt", "yuv420p")
    ffmpeg_params = ["-pix_fmt", pix_fmt] if pix_fmt else None

    frames = np.asarray(video_frames, dtype=np.uint8)
    with imageio.get_writer(
        str(path),
        fps=resolved_fps,
        codec=codec,
        format="FFMPEG",
        ffmpeg_params=ffmpeg_params,
    ) as writer:
        for frame in frames:
            writer.append_data(frame)


def show_video(video_path: str | Path, video_width: int = 400) -> HTML:
    """Embed a local MP4 in Jupyter output.

    Args:
        video_path: Path to an MP4 file on disk.
        video_width: HTML video element width in pixels.

    Returns:
        An ``IPython.display.HTML`` object with a base64 data-URL, or empty
        HTML if the file does not exist.
    """
    from base64 import b64encode

    from IPython.display import HTML

    path = Path(video_path)
    if not path.is_file():
        logging.warning("Video file not found: %s", path)
        return HTML("")

    data = path.read_bytes()
    video_url = f"data:video/mp4;base64,{b64encode(data).decode()}"
    return HTML(
        f'<video autoplay width={video_width} controls><source src="{video_url}"></video>'
    )

---
## Part 1 — Bodies and Geoms

### Concept: the world body

Every MuJoCo model has a **world body** — the fixed root of the kinematic tree.  
All other bodies are children (or grandchildren, etc.) of the world body.

A **body** is a rigid frame. On its own it is invisible; you attach **geoms** to it to give it shape (for rendering and collision).

```
worldbody  (fixed)
 └── upper_arm   (body)
      └── cylinder geom   (shape)
```

### The `MjSpec` workflow

```
spec  = MjSpec()         # mutable model specs
model = spec.compile()   # compile to a runnable MjModel
data  = MjData(model)    # mutable simulation state

# After adding more elements:
model, data = spec.recompile(model, data)   # update in-place
```

In [ ]:
UPPER_LEN = 0.28   # length of upper arm segment  [m]
RADIUS    = 0.035  # cylinder radius               [m]

# ── Create the spec and add a single upper-arm body ────────────────────────
spec = mujoco.MjSpec()

upper_body = spec.worldbody.add_body()
upper_body.name = "upper_arm"
upper_body.pos  = [0, 0, 0]

g_upper = upper_body.add_geom()
g_upper.type = mujoco.mjtGeom.mjGEOM_CYLINDER
g_upper.size = [RADIUS, UPPER_LEN / 2, 0]
g_upper.pos  = [0, 0, -UPPER_LEN / 2]
g_upper.rgba = [0.2, 0.8, 0.1, 1]

# ── Compile the spec into a runnable model ─────────────────────────────────
model = spec.compile()
data  = mujoco.MjData(model)

print(f"Bodies: {model.nbody} (incl. world)   Geoms: {model.ngeom}   DoFs: {model.nv}")

In [ ]:
# Simulation run (1 second)
duration = 1
frames = []
mujoco.mj_resetData(model, data)

with mujoco.Renderer(model) as renderer:
  while data.time < duration:
    mujoco.mj_step(model, data)
    renderer.update_scene(data)
    frames.append(renderer.render())

media.show_video(frames, fps=1/model.opt.timestep)

You should see a green cylinder in the MuJoCo environment.  
It is **static** — as there are no joints, gravity has no effect onto it.


---
## Part 2 — Adding a Second Segment

We **extend the same `spec`** by adding a forearm body as a child of `upper_body`.  
Then we call `spec.recompile()` to update the model and data objects.

```
worldbody
 └── upper_arm
      ├── cylinder geom
      └── forearm          ← ADD
           └── cylinder geom
```

In [ ]:
FORE_LEN  = 0.25   # length of forearm segment    [m]
GAP       = 0.01   # gap between segment ends to avoid self-contact [m]

# ── Extend the spec: add the forearm body ──────────────────────────────────
# A small GAP between segment endpoints avoids self-contact at the joint.

if spec.body("forearm") is None:
  forearm_body = upper_body.add_body()
forearm_body.name = "forearm"
forearm_body.pos  = [0, 0, -(UPPER_LEN + GAP)]   # offset from parent origin
forearm_body.quat = [1., 0., 1, 0.]  # 90° flexion by default 

g_fore = forearm_body.add_geom()
g_fore.type = mujoco.mjtGeom.mjGEOM_CYLINDER
g_fore.size = [RADIUS * 0.85, FORE_LEN / 2, 0]
g_fore.pos  = [0, 0, -FORE_LEN / 2]
g_fore.rgba = [0.8, 0.8, 0.1, 1]

# ── Recompile: updates model and data in-place ─────────────────────────────
model, data = spec.recompile(model, data)

print(f"Bodies: {model.nbody}   Geoms: {model.ngeom}   DoFs: {model.nv}")


# Simulation run (1 second)
duration = 1
frames = []
mujoco.mj_resetData(model, data)

with mujoco.Renderer(model) as renderer:
  while data.time < duration:
    mujoco.mj_step(model, data)
    renderer.update_scene(data)
    frames.append(renderer.render())

media.show_video(frames, fps=1/model.opt.timestep)

Both segments are rigidly welded.  
To allow for movement, we need to add a **joint**.

---
## Part 3 — Unlocking Motion with Joints

### Concept: joints

A **joint** is added to a body and defines what motion is allowed between that body and its parent.  
Without a joint the body is welded. Adding a joint *removes* constraints and introduces degrees of freedom.

| Joint type         | DoFs | Motion                               |
|--------------------|------|--------------------------------------|
| `mjJNT_HINGE`      | 1    | Rotation about one axis (pivot)      |
| `mjJNT_SLIDE`      | 1    | Translation along one axis           |
| `mjJNT_BALL`       | 3    | Unconstrained rotation (ball socket) |
| `mjJNT_FREE`       | 6    | Fully free floating body             |

We add a **hinge joint** at the elbow:

```
worldbody
 └── upper_arm
      ├── cylinder geom
      └── forearm
           ├── [hinge joint @ elbow] ← ADD
           └── cylinder geom
```

In [ ]:
# # ── Extend the spec: add hinge joint ─────────────────────────────────────
# # We keep references to these joint objects so we can modify them later.

if spec.joint("elbow") is None:
  elbow_jnt = forearm_body.add_joint()
elbow_jnt.name = "elbow"
elbow_jnt.type = mujoco.mjtJoint.mjJNT_HINGE
elbow_jnt.pos  = [0, 0, 0]      # at the proximal end of the forearm
elbow_jnt.axis = [0, 1, 0]

model, data = spec.recompile(model, data)

print(f"Bodies: {model.nbody}   DoFs: {model.nv}")


# Simulation run (3 seconds)
duration = 3
frames = []
mujoco.mj_resetData(model, data)

with mujoco.Renderer(model) as renderer:
  while data.time < duration:
    mujoco.mj_step(model, data)
    renderer.update_scene(data)
    frames.append(renderer.render())

media.show_video(frames, fps=1/model.opt.timestep)

The arm swings down and oscillates, as gravity is now acting on the forearm arm through the elbow joint.

---
## Part 4 — Joint Limits and Damping

A real elbow can't hyperextend and soft tissue resists rapid motion.  
We model this with **joint limits** (`limited`, `range`) and **damping**.

Because we kept references to `elbow_jnt`, we can modify the joint directly and recompile — no need to recreate anything.

> **Angle convention:** `MjSpec` stores joint angles in **degrees**. Pass raw degree values — no `np.deg2rad()` needed.

In [ ]:
# ── Modify the existing joint in-place ────────────────────────────────────

elbow_jnt.limited = True
elbow_jnt.range   = np.array([0, 145]) - 90    # 0–145° flexion only (degrees) [-90 subtraction because forearm was defined in 90° flexed pose]
elbow_jnt.damping = np.array([0.3, 0, 0])

model, data = spec.recompile(model, data)

print(f"DoFs: {model.nv}")
print(f"Elbow range: {np.rad2deg(model.jnt_range[0]) + 90}°  (verified in compiled model)")

# Simulation run (3 seconds)
duration = 3
frames = []
mujoco.mj_resetData(model, data)

with mujoco.Renderer(model) as renderer:
  while data.time < duration:
    mujoco.mj_step(model, data)
    renderer.update_scene(data)
    frames.append(renderer.render())

media.show_video(frames, fps=1/model.opt.timestep)

The arm now settles into a damped equilibrium; the elbow cannot hyperextend.

---
## Part 5 — Driving the Arm with a Motor Actuator

### Concept: actuators

An **actuator** converts a control signal (`data.ctrl[i]`) into a force or torque.  
The simplest type is a **motor** that applies torque directly to a joint.

```
data.ctrl[0] = 1.0  →  elbow_motor  →  torque on elbow joint
```

In [ ]:
# # ── Extend the spec: add a motor actuator on the elbow ─────────────────────
if spec.actuator("elbow_motor") is None:
  motor_act = spec.add_actuator()
motor_act.name        = "elbow_motor"
motor_act.target      = "elbow"              # acts on the joint named 'elbow'
motor_act.trntype     = mujoco.mjtTrn.mjTRN_JOINT
motor_act.gaintype    = mujoco.mjtGain.mjGAIN_FIXED
motor_act.gainprm     = np.array([30.0] + [0.0] * 9)  # torque scale [Nm per unit ctrl]
motor_act.biastype    = mujoco.mjtBias.mjBIAS_NONE
motor_act.ctrlrange   = np.array([-1.0, 1.0])
motor_act.ctrllimited = True

model, data = spec.recompile(model, data)

print(f"DoFs: {model.nv}   Actuators: {model.nu}")
print("ctrl[0] → elbow_motor")


# Simulation run (3 seconds -- first second: no torque, then full torque)
duration = 3
frames = []
mujoco.mj_resetData(model, data)

with mujoco.Renderer(model) as renderer:
  while data.time < duration:
    data.ctrl[:] = 1.0  if data.time > 1.0 else 0.0
    mujoco.mj_step(model, data)
    renderer.update_scene(data)
    frames.append(renderer.render())

media.show_video(frames, fps=1/model.opt.timestep)

The forearm is actively driven upward by the elbow motor.

---
## Part 6 — Muscle-Like Actuators

### Concept: muscles in MuJoCo

MuJoCo's built-in **muscle model** captures key physiological properties:
- **Unidirectional force** — muscles can only pull, not push
- **Force–length curve** — weaker when too short or too stretched
- **Force–velocity curve** — weaker when contracting fast
- **Activation dynamics** — the neural input is low-pass filtered before converted into force

Muscles act via **tendons** — paths connecting **sites** (marker points) on two bodies.

```
upper_arm  →  [bicep_origin site]  ── tendon ──  [bicep_insertion site]  ←  forearm
                                          ↑
                                   bicep muscle actuator
```

**Site placement** determines the muscle moment arm. For a flexor, both sites must be on the **same side** of both segments, placed so the tendon **shortens** when the elbow flexes.

In [ ]:
# ── Remove the motor (we replace it with a muscle) ─────────────────────────
spec.delete(motor_act)

# ── Add muscle sites to the existing bodies ────────────────────────────────
MOMENT_ARM = RADIUS * 2.0   # perpendicular distance from joint axis to tendon [m]

# Origin on the posterior face of the upper arm
origin_site = upper_body.add_site()
origin_site.name = "bicep_origin"
origin_site.pos  = [-MOMENT_ARM, 0, -UPPER_LEN * 0.15]
origin_site.size = [0.012, 0.012, 0.012]
origin_site.rgba = [1.0, 0.2, 0.2, 1]

# Insertion on the same (posterior) face of the forearm, near the elbow
insertion_site = forearm_body.add_site()
insertion_site.name = "bicep_insertion"
insertion_site.pos  = [-MOMENT_ARM, 0, -FORE_LEN * 0.15]
insertion_site.size = [0.012, 0.012, 0.012]
insertion_site.rgba = [1.0, 0.2, 0.2, 1]

# ── Tendon: straight-line path between origin and insertion ────────────────
bicep_tendon = spec.add_tendon()
bicep_tendon.name = "bicep_tendon"
bicep_tendon.wrap_site("bicep_origin")
bicep_tendon.wrap_site("bicep_insertion")

# ── Muscle actuator ────────────────────────────────────────────────────────
# gainprm / biasprm (10 slots): [lmin, lmax, force(-1=auto), scale, FL/FV params...]
# dynprm [0,1]: activation / deactivation time constants (seconds)
MUSCLE_PARAMS = np.array([0.75, 1.05, -1., 200., 0.5, 1.6, 1.5, 1.3, 1.2, 0.])
MUSCLE_DYN    = np.array([0.01, 0.04] + [0.0] * 8)  # 10 ms / 40 ms

bicep_act = spec.add_actuator()
bicep_act.name        = "bicep"
bicep_act.target      = "bicep_tendon"
bicep_act.trntype     = mujoco.mjtTrn.mjTRN_TENDON
bicep_act.dyntype     = mujoco.mjtDyn.mjDYN_MUSCLE
bicep_act.gaintype    = mujoco.mjtGain.mjGAIN_MUSCLE
bicep_act.biastype    = mujoco.mjtBias.mjBIAS_MUSCLE
bicep_act.gainprm     = MUSCLE_PARAMS
bicep_act.biasprm     = MUSCLE_PARAMS
bicep_act.dynprm      = MUSCLE_DYN
bicep_act.ctrlrange   = np.array([0.0, 1.0])   # 0 = relaxed, 1 = fully activated
bicep_act.ctrllimited = True

# Also shorten the cylinder geoms slightly so they don't touch at the joint
g_upper.size = [RADIUS, UPPER_LEN / 2 - 0.01, 0]
g_upper.pos  = [0, 0, -UPPER_LEN / 2 + 0.01]
g_fore.size  = [RADIUS * 0.85, FORE_LEN / 2 - 0.01, 0]
g_fore.pos   = [0, 0, -FORE_LEN / 2 + 0.01]

model, data = spec.recompile(model, data)

print(f"Bodies: {model.nbody}   DoFs: {model.nv}   "
      f"Tendons: {model.ntendon}   Actuators: {model.nu}")

In [ ]:
# Relaxed muscle — gravity drops the forearm to its lower limit

# Simulation run (3 seconds)
duration = 3
frames = []
mujoco.mj_resetData(model, data)

with mujoco.Renderer(model) as renderer:
  while data.time < duration:
    data.ctrl[0] = 0.0
    mujoco.mj_step(model, data)
    renderer.update_scene(data)
    frames.append(renderer.render())

media.show_video(frames, fps=1/model.opt.timestep)

In [ ]:
# Activated muscle — bicep contracts and flexes the elbow

# Simulation run (3 seconds)
duration = 3
frames = []
mujoco.mj_resetData(model, data)

with mujoco.Renderer(model) as renderer:
  while data.time < duration:
    data.ctrl[0] = 1.0 if data.time > 1.0 else 0.0
    mujoco.mj_step(model, data)
    renderer.update_scene(data)
    frames.append(renderer.render())

media.show_video(frames, fps=1/model.opt.timestep)

The forearm flexes as the bicep shortens.  
Notice the slower response compared to the motor in Part 5 — that is the **activation dynamics** (10 ms time constant) at work.

### Activation sweep

In [ ]:
import matplotlib.pyplot as plt

# In the incremental model: qpos[0] = shoulder, qpos[1] = elbow
elbow_qpos_idx = model.joint("elbow").qposadr[0]

ctrl_vals = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
eq_angles = []
for c in ctrl_vals:
    d = mujoco.MjData(model)
    d.ctrl[0] = c
    for _ in range(5000):
        mujoco.mj_step(model, d)
    eq_angles.append(np.rad2deg(d.qpos[elbow_qpos_idx]) + 90)  # convert to degrees and add 90° offset
    print(f"  ctrl = {c:.1f}  →  elbow = {eq_angles[-1]:.1f}°")

plt.figure(figsize=(5, 3))
plt.plot(ctrl_vals, eq_angles, 'o-', color='steelblue')
plt.xlabel("Activation (ctrl)")
plt.ylabel("Equilibrium elbow angle (°)")
plt.title("Muscle activation vs. equilibrium angle")
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

---
## Summary

| Concept | Role | MjSpec call |
|---------|------|-------------|
| **Body** | Rigid frame in kinematic tree | `parent.add_body()` |
| **Geom** | Shape for rendering & collision | `body.add_geom()` |
| **Joint** | Degrees of freedom between bodies | `body.add_joint()` |
| **Site** | Named marker point on a body | `body.add_site()` |
| **Tendon** | Geometric path through sites | `spec.add_tendon()` + `wrap_site()` |
| **Actuator** | Motor or muscle force generator | `spec.add_actuator()` |

**Key takeaways**
- Without a joint, a body is **welded** — gravity has no effect.
- Each Part extended the **same `spec`** object; `spec.recompile()` updated the compiled model.
- `MjSpec` uses **degrees** for joint angles — pass raw degree values.
- Muscles are actuators with activation dynamics acting via tendons; site geometry affects the functionality of the muscle (e.g. flexion vs. extension).